# Crypto XAI audit: run the full pipeline (Colab)

Reproduces every table and figure of the report from a clean checkout.

1. Set `REPO_URL` below to your GitHub copy of this repo (or upload the repo zip to Colab / Google Drive and set `LOCAL_ZIP`).
2. *Runtime → Run all*. The quick smoke test takes about 1 minute. The full run takes about 6–10 minutes on a 2-vCPU Colab VM.

Data are the Coin Metrics Community CSVs shipped in `data/raw/`. Nothing is downloaded at run time.

In [ ]:
REPO_URL = "https://github.com/arushiSH5-cyber/crypto-xai-audit.git"   # <- edit me
LOCAL_ZIP = ""   # alternative: path to crypto-xai-audit.zip on Drive, e.g. "/content/drive/MyDrive/crypto-xai-audit.zip"
import os, subprocess
if not os.path.exists("crypto-xai-audit"):
    if LOCAL_ZIP:
        subprocess.run(["unzip", "-q", LOCAL_ZIP], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
%cd crypto-xai-audit

In [ ]:
# lime 0.2.0.1 builds fine on Colab's Python 3.12. On Python 3.13+ use the workaround in HANDOFF.md.
!pip install -q -r requirements.txt

## 1. Tests (no look-ahead, fold ordering, regime threshold, metrics, TreeSHAP determinism, reproducibility)

In [ ]:
!python -m pytest -q

## 2. Quick smoke run (2 coins, 2024+; writes `results/quick/`)

In [ ]:
!python run_all.py --quick

## 3. Full run (5 coins × 11 folds × 4 models; writes `results/`)

In [ ]:
%time !python -W ignore run_all.py

## 4. Key results

In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
print(json.dumps(json.load(open("results/tables/key_numbers.json")), indent=1)[:1500])
display(pd.read_csv("results/tables/accuracy_by_regime.csv").round(3))
t = pd.read_csv("results/tables/regime_tests.csv")
display(t[["family","model","detail","metric","mean_calm","mean_volatile","diff","ci_lo","ci_hi","p_perm","q_bh","p_placebo"]].round(3))

In [ ]:
import glob
for f in sorted(glob.glob("results/figures/*.png")):
    display(Markdown(f"**{f}**")); display(Image(f, width=900))